## Fotocasa

In [22]:
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import time

# Abrir navegador
driver = uc.Chrome()
time.sleep(3)

# Un link de fotocasa
url = "https://www.fotocasa.es/en/buy/home/malaga-capital/air-conditioning-heating-terrace-lift-not-furnished/189631313/d?stc=dis-sharead-sharead&utm_medium=social-share&utm_source=sharead&utm_campaign=sharead,"  # pega el link completo aquí
driver.get(url)
time.sleep(6)

print(driver.title)
print(len(driver.page_source))

Apartments for sale in Calle María, Olletas - Sierra Blanquilla, Málaga Capital | fotocasa
1044454


In [23]:
soup = BeautifulSoup(driver.page_source, "html.parser")

# Buscar precio
precio = soup.find("span", class_="re-DetailHeader-price")
print("Precio:", precio)

# Buscar título
titulo = soup.find("h1")
print("Título:", titulo)

# Buscar features básicas
features = soup.find("ul", class_="re-DetailFeaturesList")
print("Features:", features)

# Buscar ubicación
ubicacion = soup.find("p", class_="re-DetailMap-address")
print("Ubicación:", ubicacion)

# Buscar descripción/comentario
comentario = soup.find("p", class_="re-DetailDescription-text")
print("Comentario:", comentario)

Precio: <span class="re-DetailHeader-price">185.000 €</span>
Título: <h1 class="re-DetailHeader-propertyTitle">Apartments for sale in Calle María, Olletas - Sierra Blanquilla</h1>
Features: None
Ubicación: None
Comentario: None


In [24]:
# Buscar features — probar clases alternativas
print("=== FEATURES ===")
for tag in soup.find_all(class_=lambda x: x and "Feature" in x):
    print(tag.name, tag.get("class"), "→", tag.text.strip()[:50])

print("\n=== UBICACION ===")
for tag in soup.find_all(class_=lambda x: x and "address" in str(x).lower()):
    print(tag.name, tag.get("class"), "→", tag.text.strip()[:50])

print("\n=== DESCRIPCION ===")
for tag in soup.find_all(class_=lambda x: x and "escription" in str(x).lower()):
    print(tag.name, tag.get("class"), "→", tag.text.strip()[:80])

=== FEATURES ===

=== UBICACION ===

=== DESCRIPCION ===
div ['re-DetailDescriptionContainer'] → Translate to EnglishGilmar Consulting Inmobiliario vende este acogedor apartamen
p ['re-DetailDescription', 'm-none', 'text-body-1'] → Gilmar Consulting Inmobiliario vende este acogedor apartamento ubicado en pleno 


In [25]:
# Buscar features con texto
print("=== BUSCAR M2 / HAB ===")
for tag in soup.find_all(class_=lambda x: x and "detail" in str(x).lower()):
    texto = tag.text.strip()
    if any(k in texto.lower() for k in ["m²", "hab", "baño", "planta", "ascensor"]):
        print(tag.name, tag.get("class"), "→", texto[:80])

print("\n=== BUSCAR UBICACION ===")
for tag in soup.find_all(class_=lambda x: x and "location" in str(x).lower() or x and "map" in str(x).lower()):
    print(tag.name, tag.get("class"), "→", tag.text.strip()[:80])

print("\n=== BUSCAR PRECIO ===")
for tag in soup.find_all(class_=lambda x: x and "price" in str(x).lower()):
    print(tag.name, tag.get("class"), "→", tag.text.strip()[:50])

=== BUSCAR M2 / HAB ===
body ['detail', 'br-Chrome', 'os-MacOS', 'osv-10_15_7'] → SearchAdvertiseMy alertsList your property for FreeLog in+23 Photos27 Photos185.
main ['re-ContentDetail'] → +23 Photos27 Photos185.000 €DiscardRoomsFavouriteCalculate your mortgageSuggest 
div ['re-ContentDetail-bothContainer'] → 185.000 €DiscardRoomsFavouriteCalculate your mortgageSuggest price1 bdrm.1 bathr
div ['re-ContentDetail-topContainer'] → 185.000 €DiscardRoomsFavouriteCalculate your mortgageSuggest price1 bdrm.1 bathr
div ['re-ContentDetail-topContainer--main'] → 185.000 €DiscardRoomsFavouriteCalculate your mortgageSuggest price1 bdrm.1 bathr
div ['re-DetailDescriptionContainer'] → Translate to EnglishGilmar Consulting Inmobiliario vende este acogedor apartamen
p ['re-DetailDescription', 'm-none', 'text-body-1'] → Gilmar Consulting Inmobiliario vende este acogedor apartamento ubicado en pleno 
div ['re-ContentDetail-featuresListWrapper'] → Property typeFlatAvailabilityAvailableOrientationWestCo

In [26]:
# Precio
precio_raw = soup.find("span", class_="re-DetailHeader-price")
print("Precio:", precio_raw.text.strip() if precio_raw else None)

# Título
titulo_raw = soup.find("h1", class_="re-DetailHeader-propertyTitle")
print("Título:", titulo_raw.text.strip() if titulo_raw else None)

# Features wrapper
features_raw = soup.find("div", class_="re-ContentDetail-featuresListWrapper")
print("Features:", features_raw.text.strip()[:200] if features_raw else None)

# Habitaciones y baños desde el header
header = soup.find("div", class_="re-ContentDetail-topContainer--main")
print("Header:", header.text.strip()[:150] if header else None)

# Descripción
comentario_raw = soup.find("p", class_="re-DetailDescription")
print("Comentario:", comentario_raw.text.strip()[:100] if comentario_raw else None)

Precio: 185.000 €
Título: Apartments for sale in Calle María, Olletas - Sierra Blanquilla
Features: Property typeFlatAvailabilityAvailableOrientationWestConditionGoodAge10 to 20 yearsFloor1st FloorLiftYesFurnishedNoEnergyEnergy rating label:GEmissions:999 kg CO₂ m² / yearEmissionsEnergy rating label
Header: 185.000 €DiscardRoomsFavouriteCalculate your mortgageSuggest price1 bdrm.1 bathroom41 sqm1st floorApartments for sale in Calle María, Olletas - Sierra
Comentario: Gilmar Consulting Inmobiliario vende este acogedor apartamento ubicado en pleno Distrito Centro de M


In [27]:
def scrape_fotocasa(driver, url):
    driver.get(url)
    time.sleep(6)

    soup = BeautifulSoup(driver.page_source, "html.parser")

    # Detectar dado de baja
    texto_pagina = soup.get_text().lower()
    if "this listing is no longer available" in texto_pagina or "ya no está disponible" in texto_pagina:
        return {
            "url": url,
            "plataforma": "fotocasa",
            "estado_anuncio": "dado de baja",
            "titulo": None, "ubicacion": None, "precio": None,
            "m2": None, "habitaciones": None, "baños": None,
            "planta": None, "ascensor": None, "tipo": None,
            "estado": None, "año": None, "anunciante": None,
            "comentario": None
        }

    try:
        precio_text = soup.find("span", class_="re-DetailHeader-price").text.strip()
        precio = int(precio_text.replace(".", "").replace("€", "").strip())
    except:
        precio = None

    try:
        titulo = soup.find("h1", class_="re-DetailHeader-propertyTitle").text.strip()
    except:
        titulo = None

    try:
        comentario = soup.find("p", class_="re-DetailDescription").text.strip()
    except:
        comentario = None

    try:
        features_text = soup.find("div", class_="re-ContentDetail-featuresListWrapper").text.strip()
    except:
        features_text = ""

    try:
        header_text = soup.find("div", class_="re-ContentDetail-topContainer--main").text.strip()
    except:
        header_text = ""

    # m2
    m2_match = re.search(r'(\d+)\s*sqm', header_text)
    m2 = int(m2_match.group(1)) if m2_match else None

    # Habitaciones
    hab_match = re.search(r'(\d+)\s*bdrm', header_text)
    habitaciones = int(hab_match.group(1)) if hab_match else None

    # Baños
    ban_match = re.search(r'(\d+)\s*bath', header_text)
    baños = int(ban_match.group(1)) if ban_match else None

    # Planta
    planta_match = re.search(r'(\d+(?:st|nd|rd|th)?\s*[Ff]loor)', header_text)
    planta = planta_match.group(1) if planta_match else None

    # Ascensor
    ascensor = "Sí" if "LiftYes" in features_text else "No" if "LiftNo" in features_text else None

    # Estado
    estado = None
    if "Good" in features_text:
        estado = "Buen estado"
    elif "New" in features_text:
        estado = "Nuevo"
    elif "Renovated" in features_text:
        estado = "Reformado"

    # Año
    age_match = re.search(r'Age(\d+)\s*to\s*(\d+)\s*years', features_text)
    año = f"{age_match.group(1)}-{age_match.group(2)} años" if age_match else None

    # Tipo
    tipo = "Casa" if titulo and ("house" in titulo.lower() or "chalet" in titulo.lower()) else "Piso"

    # Ubicación
    ubicacion = titulo.split("in ")[-1] if titulo and "in " in titulo else titulo

    # Anunciante
    try:
        anunciante = soup.find("p", class_="re-ContactDetail-name").text.strip()
    except:
        anunciante = None

    return {
        "url": url,
        "plataforma": "fotocasa",
        "estado_anuncio": "activo",
        "titulo": titulo,
        "ubicacion": ubicacion,
        "precio": precio,
        "m2": m2,
        "habitaciones": habitaciones,
        "baños": baños,
        "planta": planta,
        "ascensor": ascensor,
        "tipo": tipo,
        "estado": estado,
        "año": año,
        "anunciante": anunciante,
        "comentario": comentario
    }

In [28]:
piso = scrape_fotocasa(driver, "https://www.fotocasa.es/en/buy/home/malaga-capital/air-conditioning/189689819/d?stc=dis-sharead-sharead&utm_medium=social-share&utm_source=sharead&utm_campaign=sharead,")
for k, v in piso.items():
    print(f"{k}: {v}")

url: https://www.fotocasa.es/en/buy/home/malaga-capital/air-conditioning/189689819/d?stc=dis-sharead-sharead&utm_medium=social-share&utm_source=sharead&utm_campaign=sharead,
plataforma: fotocasa
estado_anuncio: activo
titulo: Flat for sale in Avenida de la Paloma, Girón - Las Delicias
ubicacion: Avenida de la Paloma, Girón - Las Delicias
precio: 215000
m2: 61
habitaciones: 3
baños: 1
planta: 15th floor
ascensor: No
tipo: Piso
estado: None
año: 70-100 años
anunciante: None
comentario: Se vende piso en una excelente ubicación, en Avenida de la Paloma, a tan solo 600 metros del Paseo Marítimo Antonio Banderas y muy cerca de todos los servicios, comercios y transporte público.La vivienda cuenta con:3 habitaciones1 baño completo61 m2 construidosPrimera planta SIN ascensorDistribución cómoda y funcional, ideal tanto como vivienda habitual como inversión.Zona muy demandada, perfecta para disfrutar de la cercanía a la playa y del ambiente de Málaga capital.Si lo sueñas, ¡hazlo realidad!¡No pie

### Pisos.com

In [29]:
driver = uc.Chrome()
time.sleep(3)

url = "https://www.pisos.com/alquilar/estudio-centro_historico_la_merced29008-60839514551_100500/"
driver.get(url)
time.sleep(6)

print(driver.title)
print(len(driver.page_source))

Estudio en alquiler en Centro Histórico en Centro Histórico-La Merced por 750 €/mes
262731


In [30]:
# Precio
precio_raw = soup.find("div", class_="price__value")
print("Precio:", precio_raw.text.strip() if precio_raw else None)

# Título
print("Título:", soup.find("h1").text.strip())

# Buscar m2, habitaciones, características
print("\n=== CARACTERISTICAS ===")
for tag in soup.find_all(class_=lambda x: x and "charact" in str(x).lower()):
    print(tag.name, tag.get("class"), "→", tag.text.strip()[:100])

# Buscar lista de detalles
print("\n=== DETALLES ===")
for tag in soup.find_all(class_=lambda x: x and "detail" in str(x).lower()):
    texto = tag.text.strip()
    if any(k in texto.lower() for k in ["m²", "hab", "baño", "planta", "ascensor", "piso"]):
        print(tag.name, tag.get("class"), "→", texto[:100])

Precio: None
Título: Apartments for sale in Calle María, Olletas - Sierra Blanquilla

=== CARACTERISTICAS ===

=== DETALLES ===
body ['detail', 'br-Chrome', 'os-MacOS', 'osv-10_15_7'] → SearchAdvertiseMy alertsList your property for FreeLog in+23 Photos27 Photos185.000 €DiscardRoomsFav
main ['re-ContentDetail'] → +23 Photos27 Photos185.000 €DiscardRoomsFavouriteCalculate your mortgageSuggest price1 bdrm.1 bathro
div ['re-ContentDetail-bothContainer'] → 185.000 €DiscardRoomsFavouriteCalculate your mortgageSuggest price1 bdrm.1 bathroom41 sqm1st floorAp
div ['re-ContentDetail-topContainer'] → 185.000 €DiscardRoomsFavouriteCalculate your mortgageSuggest price1 bdrm.1 bathroom41 sqm1st floorAp
div ['re-ContentDetail-topContainer--main'] → 185.000 €DiscardRoomsFavouriteCalculate your mortgageSuggest price1 bdrm.1 bathroom41 sqm1st floorAp
div ['re-DetailDescriptionContainer'] → Translate to EnglishGilmar Consulting Inmobiliario vende este acogedor apartamento ubicado en pleno 
p ['re-Detail

In [31]:
# Buscar por texto que contenga m²
print("=== BUSCAR M2 ===")
for tag in soup.find_all(string=re.compile(r'\d+\s*m²')):
    print(repr(tag.strip()[:80]))

# Buscar por texto habitaciones
print("\n=== BUSCAR HAB ===")
for tag in soup.find_all(string=re.compile(r'habitaci|dormitor|baño|ascensor|planta', re.IGNORECASE)):
    print(repr(tag.strip()[:80]))

# Buscar descripción
print("\n=== DESCRIPCION ===")
desc = soup.find("div", class_="description-modal__text")
if not desc:
    desc = soup.find("div", class_="js-description")
if not desc:
    for tag in soup.find_all("div"):
        if "descripci" in tag.text.lower()[:20]:
            print(tag.get("class"), "→", tag.text.strip()[:100])
            break
print(desc.text.strip()[:100] if desc else "No encontrado")

=== BUSCAR M2 ===
'window.__INITIAL_DATA__ = JSON.parse(\'{"browser":{"deviceType":"desktop","isAndr'

=== BUSCAR HAB ===
'La vivienda se ubica en un edificio de reciente construcción, concretamente del '
'La propiedad dispone de 41 metros cuadrados construidos y presenta una distribuc'
'{"__IMAGES_TO_PRELOAD__":[{"as":"image","href":"https://static.fotocasa.es/image'

=== DESCRIPCION ===
No encontrado


In [32]:
# Precio
precio_raw = soup.find("span", class_="re-DetailHeader-price")
print("Precio:", precio_raw.text.strip() if precio_raw else None)

# Título
titulo_raw = soup.find("h1", class_="re-DetailHeader-propertyTitle")
print("Título:", titulo_raw.text.strip() if titulo_raw else None)

# Features wrapper
features_raw = soup.find("div", class_="re-ContentDetail-featuresListWrapper")
print("Features:", features_raw.text.strip()[:200] if features_raw else None)

# Habitaciones y baños desde el header
header = soup.find("div", class_="re-ContentDetail-topContainer--main")
print("Header:", header.text.strip()[:150] if header else None)

# Descripción
comentario_raw = soup.find("p", class_="re-DetailDescription")
print("Comentario:", comentario_raw.text.strip()[:100] if comentario_raw else None)

Precio: 185.000 €
Título: Apartments for sale in Calle María, Olletas - Sierra Blanquilla
Features: Property typeFlatAvailabilityAvailableOrientationWestConditionGoodAge10 to 20 yearsFloor1st FloorLiftYesFurnishedNoEnergyEnergy rating label:GEmissions:999 kg CO₂ m² / yearEmissionsEnergy rating label
Header: 185.000 €DiscardRoomsFavouriteCalculate your mortgageSuggest price1 bdrm.1 bathroom41 sqm1st floorApartments for sale in Calle María, Olletas - Sierra
Comentario: Gilmar Consulting Inmobiliario vende este acogedor apartamento ubicado en pleno Distrito Centro de M


In [33]:
def scrape_pisos(driver, url):
    driver.get(url)
    time.sleep(6)

    soup = BeautifulSoup(driver.page_source, "html.parser")

    # Detectar dado de baja
    texto_pagina = soup.get_text().lower()
    if "anuncio no disponible" in texto_pagina or "ya no está disponible" in texto_pagina:
        return {
            "url": url, "plataforma": "pisos.com",
            "estado_anuncio": "dado de baja",
            "titulo": None, "ubicacion": None, "precio": None,
            "m2": None, "habitaciones": None, "baños": None,
            "planta": None, "ascensor": None, "tipo": None,
            "estado": None, "año": None, "anunciante": None,
            "comentario": None
        }

    # Precio
    try:
        precio_text = soup.find("div", class_="price__value").text.strip()
        precio = int(re.search(r'[\d\.]+', precio_text).group().replace(".", ""))
    except:
        precio = None

    # Título
    try:
        titulo = soup.find("h1").text.strip()
    except:
        titulo = None

    # Features — todos los items de la lista
    try:
        items = soup.find("ul", class_="features-summary").find_all("li", class_="features-summary__item")
        features_texts = [item.text.strip() for item in items]
    except:
        features_texts = []

    # Parsear features
    m2, habitaciones, baños, planta, ascensor = None, None, None, None, None
    for feat in features_texts:
        feat_lower = feat.lower()
        if "m²" in feat:
            m2_match = re.search(r'(\d+)\s*m²', feat)
            if m2_match:
                m2 = int(m2_match.group(1))
        elif "habitaci" in feat_lower or "dormitor" in feat_lower:
            hab_match = re.search(r'(\d+)', feat)
            if hab_match:
                habitaciones = int(hab_match.group(1))
        elif "baño" in feat_lower:
            ban_match = re.search(r'(\d+)', feat)
            if ban_match:
                baños = int(ban_match.group(1))
        elif "planta" in feat_lower:
            planta = feat
        elif "ascensor" in feat_lower:
            ascensor = "Sí" if "sin ascensor" not in feat_lower else "No"

    # Tipo
    tipo = "Casa" if titulo and any(k in titulo.lower() for k in ["casa", "chalet", "villa"]) else "Piso"

    # Ubicación desde título
    try:
        ubicacion = soup.find("span", class_="show-map__address").text.strip()
    except:
        ubicacion = titulo.split(" en ")[-1] if titulo and " en " in titulo else titulo

    # Anunciante
    try:
        anunciante = soup.find("div", class_="advertiser__name").text.strip()
    except:
        anunciante = None

    # Comentario
    try:
        comentario = soup.find("div", class_="description-modal__text").text.strip()
        if not comentario:
            comentario = soup.find("div", class_="js-description").text.strip()
    except:
        comentario = None

    # Estado y año — desde features extendidas
    estado, año = None, None
    try:
        extra_items = soup.find_all("span", class_="features__value")
        for item in extra_items:
            texto = item.text.strip().lower()
            if "buen estado" in texto or "reformado" in texto or "nuevo" in texto:
                estado = item.text.strip()
            if re.search(r'\d{4}', texto):
                año_match = re.search(r'\d{4}', texto)
                if año_match and 1900 < int(año_match.group()) < 2026:
                    año = int(año_match.group())
    except:
        pass

    return {
        "url": url,
        "plataforma": "pisos.com",
        "estado_anuncio": "activo",
        "titulo": titulo,
        "ubicacion": ubicacion,
        "precio": precio,
        "m2": m2,
        "habitaciones": habitaciones,
        "baños": baños,
        "planta": planta,
        "ascensor": ascensor,
        "tipo": tipo,
        "estado": estado,
        "año": año,
        "anunciante": anunciante,
        "comentario": comentario
    }

# ── LOOP PISOS.COM ───────────────────────────────────
driver = uc.Chrome()
time.sleep(3)

resultados_pisos = []
for i, url in enumerate(lista_pisos):
    print(f"\nScraping {i+1}/{len(lista_pisos)}: {url}")
    try:
        piso = scrape_pisos(driver, url)
        resultados_pisos.append(piso)
        print(f"✓ {piso['estado_anuncio']} — {piso['ubicacion']} — {piso['precio']}€")
    except Exception as e:
        print(f"✗ Error: {e}")
    espera = random.uniform(4, 8)
    print(f"Esperando {espera:.1f}s...")
    time.sleep(espera)

os.makedirs("data", exist_ok=True)
df_pisos = pd.DataFrame(resultados_pisos)
df_pisos.to_csv("data/pisos_scraped.csv", index=False)
print(f"\n✅ Pisos.com listo — {len(resultados_pisos)} pisos")
print(df_pisos["estado_anuncio"].value_counts())

driver.quit()

NameError: name 'lista_pisos' is not defined

### YaEncontre

In [37]:
driver = uc.Chrome()
time.sleep(3)

url = "https://www.yaencontre.com/venta/piso/inmueble-52152-111290263"
driver.get(url)
time.sleep(6)

print(driver.title)
print(len(driver.page_source))

Piso en El Ejido - La Merced - La Victoria, Málaga · 111290263 - yaencontre
140535


In [39]:
soup = BeautifulSoup(driver.page_source, "html.parser")

# Precio
print("=== PRECIO ===")
for tag in soup.find_all(class_=lambda x: x and "price" in str(x).lower()):
    print(tag.name, tag.get("class"), "→", tag.text.strip()[:60])

# Título
print("\n=== TITULO ===")
print(soup.find("h1").text.strip() if soup.find("h1") else None)

# Features
print("\n=== FEATURES ===")
for tag in soup.find_all(class_=lambda x: x and "feature" in str(x).lower()):
    print(tag.name, tag.get("class"), "→", tag.text.strip()[:80])

# Descripción
print("\n=== DESCRIPCION ===")
for tag in soup.find_all(class_=lambda x: x and "desc" in str(x).lower()):
    print(tag.name, tag.get("class"), "→", tag.text.strip()[:80])

=== PRECIO ===
div ['previousPriceIcon', 'icon-arrow-down-round'] → 
span ['icon-text', 'previousPriceText'] → 209.000 €
div ['lower-price-wrapper', 'flex'] → Avísame si bajaCalcula tu hipoteca
div ['form-errors', 'minimum-price-error'] → Los bancos no suelen dar hipotecas para importes inferiores 

=== TITULO ===
Venta de piso en El Ejido - La Merced - La Victoria de 2 habitaciones con terraza y aire acondicionado

=== FEATURES ===
li ['hIcon', 'feature'] → Aire acondicionado
li ['hIcon', 'feature'] → Calefacción
li ['hIcon', 'feature'] → Terraza
li ['feature'] → Aire acondicionado: otros
li ['feature'] → Año de construcción: 1965
li ['feature'] → Combustible calefacción: electricidad
li ['feature'] → Estado: buen estado
li ['feature'] → Interior / exterior: exterior
li ['feature'] → Orientado a: Sur
li ['feature'] → Planta 3ª
li ['feature'] → Sistema calefacción: independiente

=== DESCRIPCION ===
section ['details-description', 'separator'] → Descripción✨ Oportunidad de inversión ex

In [40]:
# Precio
print("=== PRECIO DETALLE ===")
for tag in soup.find_all(class_=lambda x: x and "price" in str(x).lower()):
    texto = tag.text.strip()
    if "€" in texto and len(texto) < 20:
        print(tag.name, tag.get("class"), "→", texto)

# m2 y habitaciones
print("\n=== M2 Y HAB ===")
for tag in soup.find_all(class_=lambda x: x and "stat" in str(x).lower()):
    print(tag.name, tag.get("class"), "→", tag.text.strip()[:80])

# Buscar m2 directamente
print("\n=== BUSCAR M2 ===")
for tag in soup.find_all(string=re.compile(r'\d+\s*m²')):
    print(repr(tag.strip()[:60]))

# Ascensor
print("\n=== ASCENSOR ===")
for tag in soup.find_all(string=re.compile(r'ascensor|lift', re.IGNORECASE)):
    print(repr(tag.strip()[:60]))

=== PRECIO DETALLE ===
span ['icon-text', 'previousPriceText'] → 209.000 €

=== M2 Y HAB ===

=== BUSCAR M2 ===
'54 m²'

=== ASCENSOR ===


In [41]:
# Precio actual
print("=== PRECIO ACTUAL ===")
for tag in soup.find_all(class_=lambda x: x and "main" in str(x).lower()):
    texto = tag.text.strip()
    if "€" in texto and len(texto) < 30:
        print(tag.name, tag.get("class"), "→", texto)

# Buscar precio por €
print("\n=== TODOS LOS € ===")
for tag in soup.find_all(string=re.compile(r'\d+[\d\.]+\s*€')):
    print(repr(tag.strip()[:60]))

# Habitaciones
print("\n=== HABITACIONES ===")
for tag in soup.find_all(string=re.compile(r'\d+\s*hab|dormitor', re.IGNORECASE)):
    print(repr(tag.strip()[:60]))

# Contenedor principal de stats
print("\n=== STATS CONTAINER ===")
for tag in soup.find_all(class_=lambda x: x and "detail" in str(x).lower()):
    texto = tag.text.strip()
    if "m²" in texto and len(texto) < 150:
        print(tag.name, tag.get("class"), "→", texto[:100])

=== PRECIO ACTUAL ===

=== TODOS LOS € ===
'209.000 €'
'Los bancos no suelen dar hipotecas para importes inferiores '

=== HABITACIONES ===
'Venta de piso en El Ejido - La Merced - La Victoria de 2 hab'
'✨ Oportunidad de inversión excepcional ✨ CALLE LOS NEGROS \n\n'
'{"@context":"https://schema.org","@graph":[{"@type":"WebSite'

=== STATS CONTAINER ===
section ['details-header-info', 'flex', 'mb-lg'] → 199.000 €209.000 €Avísame si bajaCalcula tu hipoteca2154 m²


In [42]:
# Precio actual
try:
    header_info = soup.find("section", class_="details-header-info")
    precios = re.findall(r'[\d\.]+\s*€', header_info.text)
    precio_text = precios[0] if precios else None
    precio = int(precio_text.replace(".", "").replace("€", "").strip()) if precio_text else None
except:
    precio = None

# Título
try:
    titulo = soup.find("h1").text.strip()
except:
    titulo = None

# Features — lista de li class="feature"
try:
    features = soup.find_all("li", class_="feature")
    features_dict = {}
    for f in features:
        texto = f.text.strip()
        if ":" in texto:
            key, val = texto.split(":", 1)
            features_dict[key.strip().lower()] = val.strip()
        else:
            features_dict[texto.lower()] = True
except:
    features_dict = {}

# m2
m2_match = re.search(r'(\d+)\s*m²', header_info.text if header_info else "")
m2 = int(m2_match.group(1)) if m2_match else None

# Habitaciones desde título
hab_match = re.search(r'(\d+)\s*hab', titulo.lower() if titulo else "")
habitaciones = int(hab_match.group(1)) if hab_match else None

# Planta
planta = features_dict.get("planta", None)

# Año
año_raw = features_dict.get("año de construcción", None)
año = int(año_raw) if año_raw and año_raw.isdigit() else None

# Estado
estado = features_dict.get("estado", None)

# Ascensor
ascensor = "Sí" if "ascensor" in features_dict else "No"

# Tipo
tipo = "Casa" if titulo and any(k in titulo.lower() for k in ["casa", "chalet", "villa", "adosada"]) else "Piso"

# Ubicación
ubicacion = titulo.split(" en ")[-1].split(" de ")[0] if titulo and " en " in titulo else titulo

# Descripción
try:
    comentario = soup.find("div", class_="readMoreText").text.strip()
except:
    comentario = None

# Baños
ban_match = re.search(r'(\d+)\s*ba[ñn]', titulo.lower() if titulo else "")
baños = int(ban_match.group(1)) if ban_match else None

print(f"precio:       {precio}")
print(f"titulo:       {titulo}")
print(f"ubicacion:    {ubicacion}")
print(f"m2:           {m2}")
print(f"habitaciones: {habitaciones}")
print(f"baños:        {baños}")
print(f"planta:       {planta}")
print(f"ascensor:     {ascensor}")
print(f"estado:       {estado}")
print(f"año:          {año}")
print(f"tipo:         {tipo}")
print(f"comentario:   {comentario[:80] if comentario else None}...")
print(f"features:     {features_dict}")

precio:       199000
titulo:       Venta de piso en El Ejido - La Merced - La Victoria de 2 habitaciones con terraza y aire acondicionado
ubicacion:    El Ejido - La Merced - La Victoria
m2:           2154
habitaciones: 2
baños:        None
planta:       None
ascensor:     No
estado:       buen estado
año:          1965
tipo:         Piso
comentario:   ✨ Oportunidad de inversión excepcional ✨ CALLE LOS NEGROS 

Situado en el corazó...
features:     {'aire acondicionado': 'otros', 'calefacción': True, 'terraza': True, 'año de construcción': '1965', 'combustible calefacción': 'electricidad', 'estado': 'buen estado', 'interior / exterior': 'exterior', 'orientado a': 'Sur', 'planta 3ª': True, 'sistema calefacción': 'independiente'}


In [43]:
# m2 — buscar directamente en el texto
m2_match = re.search(r'(\d+)\s*m²', soup.get_text())
m2 = int(m2_match.group(1)) if m2_match else None

# Planta — buscar key que empiece por "planta"
planta = None
for key in features_dict:
    if key.startswith("planta"):
        planta = key  # "planta 3ª"
        break

print(f"m2:    {m2}")
print(f"planta: {planta}")

m2:    2154
planta: planta 3ª


In [47]:
# Esperar más y usar JavaScript para extraer el texto
import time

time.sleep(5)

# Extraer con JavaScript directamente
m2_js = driver.execute_script("""
    const elements = document.querySelectorAll('*');
    for (let el of elements) {
        if (el.children.length === 0 && el.textContent.includes('m²')) {
            return el.textContent.trim();
        }
    }
    return null;
""")
print("m2 JS:", m2_js)

# Extraer stats con JS
stats_js = driver.execute_script("""
    const header = document.querySelector('.details-header-info');
    return header ? header.innerText : null;
""")
print("Stats JS:", repr(stats_js[:200]) if stats_js else None)

m2 JS: 54 m²
Stats JS: '199.000\xa0€\n\n209.000 €\nCalcula tu hipoteca\n2\n1\n54 m²'


In [48]:
# Parsear stats desde JS
stats_text = stats_js.replace('\xa0', ' ')
print(repr(stats_text))

# Extraer valores
precio_match = re.search(r'([\d\.]+)\s*€', stats_text)
precio = int(precio_match.group(1).replace(".", "")) if precio_match else None

m2_match = re.search(r'(\d+)\s*m²', stats_text)
m2 = int(m2_match.group(1)) if m2_match else None

# Habitaciones y baños — los números sueltos antes del m2
numeros = re.findall(r'\b(\d+)\b', stats_text.split('m²')[0])
# Filtrar precio — quitar números grandes
numeros_pequenos = [int(n) for n in numeros if int(n) < 20]
habitaciones = numeros_pequenos[0] if len(numeros_pequenos) > 0 else None
baños = numeros_pequenos[1] if len(numeros_pequenos) > 1 else None

print(f"precio:       {precio}")
print(f"m2:           {m2}")
print(f"habitaciones: {habitaciones}")
print(f"baños:        {baños}")

'199.000 €\n\n209.000 €\nCalcula tu hipoteca\n2\n1\n54 m²'
precio:       199000
m2:           54
habitaciones: 0
baños:        0


In [49]:
# Extraer solo la parte relevante — después del precio anterior
stats_clean = stats_text.split("Calcula tu hipoteca")[-1].strip()
print("Stats clean:", repr(stats_clean))

numeros = re.findall(r'\b(\d+)\b', stats_clean.split('m²')[0])
print("Números:", numeros)

numeros_validos = [int(n) for n in numeros if 0 < int(n) < 20]
habitaciones = numeros_validos[0] if len(numeros_validos) > 0 else None
baños = numeros_validos[1] if len(numeros_validos) > 1 else None

print(f"habitaciones: {habitaciones}")
print(f"baños:        {baños}")

Stats clean: '2\n1\n54 m²'
Números: ['2', '1', '54']
habitaciones: 2
baños:        1


### TEcnocasa

In [50]:
driver = uc.Chrome()
time.sleep(3)

url = "https://www.tecnocasa.es/venta/piso/malaga/malaga/653845.html"
driver.get(url)
time.sleep(6)

print(driver.title)
print(len(driver.page_source))

Piso en venta en Málaga - Carlos De Haya, Carlos De Haya, 179.900 €, 60 m2, ref. 653845 - Tecnocasa.es
90843


In [51]:
soup = BeautifulSoup(driver.page_source, "html.parser")

# Título
print("=== TITULO ===")
print(soup.find("h1").text.strip() if soup.find("h1") else None)

# Precio
print("\n=== PRECIO ===")
for tag in soup.find_all(class_=lambda x: x and "price" in str(x).lower()):
    print(tag.name, tag.get("class"), "→", tag.text.strip()[:60])

# Features
print("\n=== FEATURES ===")
for tag in soup.find_all(class_=lambda x: x and "feature" in str(x).lower()):
    print(tag.name, tag.get("class"), "→", tag.text.strip()[:80])

# Descripción
print("\n=== DESCRIPCION ===")
for tag in soup.find_all(class_=lambda x: x and "desc" in str(x).lower()):
    print(tag.name, tag.get("class"), "→", tag.text.strip()[:80])

# M2 directo
print("\n=== M2 ===")
for tag in soup.find_all(string=re.compile(r'\d+\s*m²', re.IGNORECASE)):
    print(repr(tag.strip()[:60]))

=== TITULO ===
Piso en venta en Teatinos - Portada Alta - Carlos Haya

=== PRECIO ===
div ['estate-price'] → 179.900 €
span ['current-price'] → 179.900 €
div ['estate-price'] → 179.900 €
span ['current-price'] → 179.900 €
div ['estate-card-price'] → 149.900 €
div ['estate-card-current-price'] → 149.900 €
div ['estate-card-price'] → 149.500 €
div ['estate-card-current-price'] → 149.500 €

=== FEATURES ===
div ['col-md-12', 'estate-features'] → Características del inmueble
             REF.: 
                        653845

div ['col', 'estate-features-title'] → REF.:
div ['col', 'estate-features-value'] → 653845
div ['col', 'estate-features-title'] → Amueblado:
div ['col', 'estate-features-value'] → -
div ['col', 'estate-features-title'] → Tipo de inmueble:
div ['col', 'estate-features-value'] → Popular
div ['col', 'estate-features-title'] → Año de construcción:
div ['col', 'estate-features-value'] → 1976
div ['col', 'estate-features-title'] → Tipo de propiedad:
div ['col', 'estate-feat

In [52]:
# Precio
try:
    precio_text = soup.find("span", class_="current-price").text.strip()
    precio = int(precio_text.replace(".", "").replace("€", "").strip())
except:
    precio = None

# Título
try:
    titulo = soup.find("h1").text.strip()
except:
    titulo = None

# Features — pares título/valor
try:
    titulos = soup.find_all("div", class_="estate-features-title")
    valores = soup.find_all("div", class_="estate-features-value")
    features_dict = {}
    for t, v in zip(titulos, valores):
        key = t.text.strip().replace(":", "").lower()
        val = v.text.strip()
        features_dict[key] = val
except:
    features_dict = {}

print("Features dict:", features_dict)

# Ubicación
try:
    ubicacion = soup.find("div", class_="estate-location").text.strip()
except:
    ubicacion = titulo.split(" en ")[-1] if titulo and " en " in titulo else titulo

# Descripción
try:
    comentario = soup.find("div", class_="estate-description").text.strip()
except:
    comentario = None

# M2 desde el title de la página
m2_match = re.search(r'(\d+)\s*m2', driver.title)
m2 = int(m2_match.group(1)) if m2_match else None

print(f"precio:    {precio}")
print(f"titulo:    {titulo}")
print(f"ubicacion: {ubicacion}")
print(f"m2:        {m2}")
print(f"comentario: {comentario[:80] if comentario else None}")

Features dict: {'ref.': '653845', 'amueblado': '-', 'tipo de inmueble': 'Popular', 'año de construcción': '1976', 'tipo de propiedad': '-'}
precio:    179900
titulo:    Piso en venta en Teatinos - Portada Alta - Carlos Haya
ubicacion: Teatinos - Portada Alta - Carlos Haya
m2:        60
comentario: Descripción del inmueble
             Agencia inmobiliaria de Málaga, TECNOCASA 


In [53]:
# Buscar habitaciones, baños, planta, ascensor
print("=== BUSCAR HAB/BAÑOS ===")
for tag in soup.find_all(string=re.compile(r'habitaci|dormitor|baño|planta|ascensor', re.IGNORECASE)):
    texto = tag.strip()
    if texto and len(texto) < 60:
        print(repr(texto))

# Ver el bloque completo de features
print("\n=== FEATURES COMPLETO ===")
features_block = soup.find("div", class_="estate-features")
if features_block:
    print(features_block.text.strip()[:500])

=== BUSCAR HAB/BAÑOS ===
'1 baño'
'1 baño'
'1 baño'
'1 baño'

=== FEATURES COMPLETO ===
Características del inmueble
             REF.: 
                        653845
                    Amueblado: 
                        -
                    Tipo de inmueble: 
                        Popular
                    Año de construcción: 
                        1976
                    Tipo de propiedad: 
                        -


In [54]:
# Stats con JavaScript
stats_js = driver.execute_script("""
    const stats = document.querySelectorAll('.estate-features-icon, .estate-stat, .estate-basic-info, .property-stats');
    if (stats.length > 0) return Array.from(stats).map(s => s.innerText).join(' | ');
    
    // Buscar el bloque con dormitorios y baño
    const all = document.querySelectorAll('*');
    for (let el of all) {
        if (el.children.length === 0 && el.innerText && 
            (el.innerText.includes('dorm') || el.innerText.includes('baño'))) {
            return el.innerText.trim();
        }
    }
    return null;
""")
print("Stats JS:", stats_js)

# Buscar el contenedor de 2dorm / 60m2 / 1baño
header_js = driver.execute_script("""
    const header = document.querySelector('.estate-header, .property-header, .estate-top');
    return header ? header.innerText : null;
""")
print("Header JS:", header_js)

Stats JS: 2 dorm.
Header JS: None


In [55]:
# Buscar el contenedor completo con dorm y baño
container_js = driver.execute_script("""
    const all = document.querySelectorAll('*');
    for (let el of all) {
        if (el.innerText && el.innerText.includes('dorm') && 
            el.innerText.includes('baño') && el.innerText.includes('m²')) {
            return el.innerText.trim();
        }
    }
    return null;
""")
print("Container:", repr(container_js[:200]) if container_js else None)

Container: None


In [56]:
# Extraer cada stat por separado
stats_js = driver.execute_script("""
    const results = [];
    const all = document.querySelectorAll('*');
    for (let el of all) {
        if (el.children.length === 0 && el.innerText) {
            const t = el.innerText.trim();
            if (t.match(/\\d+\\s*(dorm|baño|m²)/i)) {
                results.push(t);
            }
        }
    }
    return results;
""")
print("Stats:", stats_js)

Stats: ['2 dorm.', '1 baño', '2 dorm.', '1 baño', '1 baño', '1 baño']


In [57]:
# Habitaciones y baños desde JS
try:
    stats_js = driver.execute_script("""
        const results = [];
        const all = document.querySelectorAll('*');
        for (let el of all) {
            if (el.children.length === 0 && el.innerText) {
                const t = el.innerText.trim();
                if (t.match(/\\d+\\s*(dorm|baño|m²)/i)) {
                    results.push(t);
                }
            }
        }
        return results;
    """)
    # Tomar solo únicos
    stats_unicos = list(dict.fromkeys(stats_js))
    
    habitaciones = None
    baños = None
    for stat in stats_unicos:
        if "dorm" in stat.lower():
            hab_match = re.search(r'(\d+)', stat)
            if hab_match:
                habitaciones = int(hab_match.group(1))
        elif "baño" in stat.lower():
            ban_match = re.search(r'(\d+)', stat)
            if ban_match:
                baños = int(ban_match.group(1))
except:
    habitaciones, baños = None, None

print(f"habitaciones: {habitaciones}")
print(f"baños:        {baños}")

habitaciones: 2
baños:        1


### Otro